In [ ]:
import os

# Directorio de destino
dataset_dir = '/home/manu/TFG2/ds003490'

# Crear directorio si no existe
os.makedirs(dataset_dir, exist_ok=True)

print(f"Descargando dataset en: {dataset_dir}")

In [ ]:
# Descargar usando openneuro-py
!openneuro-py download --dataset=ds003490 --target-dir=/home/manu/TFG2/ds003490

In [6]:
import os
import numpy as np
import mne
from scipy.signal import coherence
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [7]:
# Rutas
DATASET_PATH = '/home/manu/TFG2/ds003490'
OUTPUT_PATH = '/home/manu/TFG2/EEG-Gold-Standard-main/ConnectivityMatrices_ds003490'

# Parámetros
EPOCH_DURATION = 2.0  # segundos
SFREQ = 500  # Hz
ALPHA_BAND = (8, 12)  # Hz

# Canales a EXCLUIR (no son EEG)
EXCLUDE_CHANNELS = ['VEOG', 'X', 'Y', 'Z']

# División de sujetos
PD_SUBJECTS = ['sub-001', 'sub-002', 'sub-004', 'sub-006', 'sub-007', 'sub-008', 'sub-009', 'sub-010',
               'sub-011', 'sub-012', 'sub-013', 'sub-014', 'sub-015', 'sub-016', 'sub-017', 'sub-018',
               'sub-019', 'sub-020', 'sub-021', 'sub-022', 'sub-023', 'sub-024', 'sub-025', 'sub-026', 'sub-027']

CTL_SUBJECTS = ['sub-003', 'sub-005', 'sub-028', 'sub-029', 'sub-030', 'sub-031', 'sub-032', 'sub-033',
                'sub-034', 'sub-035', 'sub-036', 'sub-037', 'sub-038', 'sub-039', 'sub-040', 'sub-041',
                'sub-042', 'sub-043', 'sub-044', 'sub-045', 'sub-046', 'sub-047', 'sub-048', 'sub-049', 'sub-050']

# Training: primeros 20 de cada grupo
PD_TRAIN = PD_SUBJECTS[:20]  # sub-001 a sub-022 (20)
CTL_TRAIN = CTL_SUBJECTS[:20]  # sub-003, sub-005, sub-028 a sub-045 (20)

# Test: últimos 5 de cada grupo
PD_TEST = PD_SUBJECTS[20:]  # sub-023 a sub-027 (5)
CTL_TEST = CTL_SUBJECTS[20:]  # sub-046 a sub-050 (5)

print(f"PD Train: {len(PD_TRAIN)} sujetos")
print(f"CTL Train: {len(CTL_TRAIN)} sujetos")
print(f"PD Test: {len(PD_TEST)} sujetos")
print(f"CTL Test: {len(CTL_TEST)} sujetos")

PD Train: 20 sujetos
CTL Train: 20 sujetos
PD Test: 5 sujetos
CTL Test: 5 sujetos


In [3]:
# Crear directorios
for split in ['Training', 'Test']:
    for n_channels in ['64', '32', '16', '8']:
        for label in ['CTL', 'PD']:
            path = os.path.join(OUTPUT_PATH, split, n_channels, label)
            os.makedirs(path, exist_ok=True)
            
print(f"Estructura creada en: {OUTPUT_PATH}")

Estructura creada en: /home/manu/TFG2/EEG-Gold-Standard-main/ConnectivityMatrices_ds003490


In [8]:
def get_eeg_file(subject):
    """Obtener el archivo .set de resting state para un sujeto"""
    # Usar sesión 01 para todos
    file_path = os.path.join(DATASET_PATH, subject, 'ses-01', 'eeg', 
                             f'{subject}_ses-01_task-Rest_eeg.set')
    if os.path.exists(file_path):
        return file_path
    return None


def load_eeg_data(file_path, target_channels):
    """Cargar datos EEG y seleccionar canales"""
    raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
    
    # Obtener canales disponibles (excluyendo los no-EEG)
    available_channels = [ch for ch in raw.ch_names if ch not in EXCLUDE_CHANNELS]
    
    # Seleccionar los canales objetivo que estén disponibles
    channels_to_use = [ch for ch in target_channels if ch in available_channels]
    
    if len(channels_to_use) < len(target_channels):
        missing = set(target_channels) - set(channels_to_use)
        print(f"  Advertencia: faltan canales {missing}")
    
    raw.pick_channels(channels_to_use)
    return raw


def compute_coherence_matrix(data, sfreq, freq_band):
    """Calcular matriz de coherencia para una banda de frecuencia"""
    n_channels = data.shape[0]
    n_samples = data.shape[1]
    
    # nperseg para coherencia (ajustado a 500 Hz)
    nperseg = min(256, n_samples // 2)
    
    coh_matrix = np.zeros((n_channels, n_channels))
    
    for i in range(n_channels):
        for j in range(i, n_channels):
            if i == j:
                coh_matrix[i, j] = 1.0
            else:
                f, Cxy = coherence(data[i], data[j], fs=sfreq, nperseg=nperseg)
                # Seleccionar banda alpha
                idx = np.where((f >= freq_band[0]) & (f <= freq_band[1]))[0]
                if len(idx) > 0:
                    coh_value = np.mean(Cxy[idx])
                else:
                    coh_value = 0.0
                coh_matrix[i, j] = coh_value
                coh_matrix[j, i] = coh_value
    
    return coh_matrix


def apply_masks(matrix):
    """Aplicar máscaras: diagonal=0, valores<0.1=0"""
    masked = matrix.copy()
    np.fill_diagonal(masked, 0)
    masked[masked < 0.1] = 0
    return masked

In [9]:
# Los 64 canales EEG disponibles (en orden)
ALL_64_CHANNELS = [
    'Fp1', 'Fz', 'F3', 'F7', 'FT9', 'FC5', 'FC1', 'C3', 'T7', 'TP9',
    'CP5', 'CP1', 'Pz', 'P3', 'P7', 'O1', 'Oz', 'O2', 'P4', 'P8',
    'TP10', 'CP6', 'CP2', 'Cz', 'C4', 'T8', 'FT10', 'FC6', 'FC2', 'F4',
    'F8', 'Fp2', 'AF7', 'AF3', 'AFz', 'F1', 'F5', 'FT7', 'FC3', 'FCz',
    'C1', 'C5', 'TP7', 'CP3', 'P1', 'P5', 'PO7', 'PO3', 'POz', 'PO4',
    'PO8', 'P6', 'P2', 'CP4', 'TP8', 'C6', 'C2', 'FC4', 'FT8', 'F6',
    'F2', 'AF4', 'AF8'
]

# Nota: son 63 canales, añadimos uno más si hay
print(f"Canales EEG definidos: {len(ALL_64_CHANNELS)}")

# Verificar contra el archivo real
sample_file = get_eeg_file('sub-001')
raw_sample = mne.io.read_raw_eeglab(sample_file, preload=False, verbose=False)
available = [ch for ch in raw_sample.ch_names if ch not in EXCLUDE_CHANNELS]
print(f"Canales disponibles en archivo: {len(available)}")
print(f"Canales: {available}")

Canales EEG definidos: 63
Canales disponibles en archivo: 63
Canales: ['Fp1', 'Fz', 'F3', 'F7', 'FT9', 'FC5', 'FC1', 'C3', 'T7', 'TP9', 'CP5', 'CP1', 'Pz', 'P3', 'P7', 'O1', 'Oz', 'O2', 'P4', 'P8', 'TP10', 'CP6', 'CP2', 'Cz', 'C4', 'T8', 'FT10', 'FC6', 'FC2', 'F4', 'F8', 'Fp2', 'AF7', 'AF3', 'AFz', 'F1', 'F5', 'FT7', 'FC3', 'FCz', 'C1', 'C5', 'TP7', 'CP3', 'P1', 'P5', 'PO7', 'PO3', 'POz', 'PO4', 'PO8', 'P6', 'P2', 'CP4', 'TP8', 'C6', 'C2', 'FC4', 'FT8', 'F6', 'F2', 'AF4', 'AF8']


In [10]:
# Configuraciones de canales estándar (sistema 10-20)

# 8 canales - Core del sistema 10-20
CHANNELS_8 = ['F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2']

# 16 canales - Añade línea media y temporales
CHANNELS_16 = [
    'Fp1', 'Fp2', 'F3', 'F4', 'Fz',
    'C3', 'C4', 'Cz', 'T7', 'T8',
    'P3', 'P4', 'Pz', 'O1', 'O2', 'Oz'
]

# 32 canales - Añade frontocentrales y parietales
CHANNELS_32 = [
    'Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8',
    'FC5', 'FC1', 'FC2', 'FC6',
    'T7', 'C3', 'Cz', 'C4', 'T8',
    'CP5', 'CP1', 'CP2', 'CP6',
    'P7', 'P3', 'Pz', 'P4', 'P8',
    'PO3', 'PO4', 'O1', 'Oz', 'O2',
    'AF3', 'AF4'
]

# 63 canales - Todos los disponibles (excluye VEOG, X, Y, Z)
CHANNELS_63 = [
    'Fp1', 'Fz', 'F3', 'F7', 'FT9', 'FC5', 'FC1', 'C3', 'T7', 'TP9',
    'CP5', 'CP1', 'Pz', 'P3', 'P7', 'O1', 'Oz', 'O2', 'P4', 'P8',
    'TP10', 'CP6', 'CP2', 'Cz', 'C4', 'T8', 'FT10', 'FC6', 'FC2', 'F4',
    'F8', 'Fp2', 'AF7', 'AF3', 'AFz', 'F1', 'F5', 'FT7', 'FC3', 'FCz',
    'C1', 'C5', 'TP7', 'CP3', 'P1', 'P5', 'PO7', 'PO3', 'POz', 'PO4',
    'PO8', 'P6', 'P2', 'CP4', 'TP8', 'C6', 'C2', 'FC4', 'FT8', 'F6',
    'F2', 'AF4', 'AF8'
]

CHANNEL_CONFIGS = {
    '64': CHANNELS_63,  # Usamos 63, guardamos en carpeta "64" para compatibilidad
    '32': CHANNELS_32,
    '16': CHANNELS_16,
    '8': CHANNELS_8
}

for key, channels in CHANNEL_CONFIGS.items():
    print(f"{key} canales: {len(channels)} → {channels[:5]}...")

64 canales: 63 → ['Fp1', 'Fz', 'F3', 'F7', 'FT9']...
32 canales: 32 → ['Fp1', 'Fp2', 'F7', 'F3', 'Fz']...
16 canales: 16 → ['Fp1', 'Fp2', 'F3', 'F4', 'Fz']...
8 canales: 8 → ['F3', 'F4', 'C3', 'C4', 'P3']...


In [12]:
def process_subject(subject, label, split, channel_configs):
    """Procesar un sujeto y guardar matrices para todas las configuraciones de canales"""
    
    file_path = get_eeg_file(subject)
    if file_path is None:
        print(f"  {subject}: archivo no encontrado")
        return 0
    
    # Cargar datos con todos los canales EEG
    raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
    
    # Obtener canales EEG disponibles
    eeg_channels = [ch for ch in raw.ch_names if ch not in EXCLUDE_CHANNELS]
    raw.pick_channels(eeg_channels)
    
    # Obtener datos
    data = raw.get_data()
    sfreq = raw.info['sfreq']
    
    # Calcular número de epochs
    samples_per_epoch = int(EPOCH_DURATION * sfreq)
    n_epochs = data.shape[1] // samples_per_epoch
    
    matrices_count = 0
    
    for epoch_idx in range(n_epochs):
        start = epoch_idx * samples_per_epoch
        end = start + samples_per_epoch
        epoch_data = data[:, start:end]
        
        # Para cada configuración de canales
        for n_ch, channels in channel_configs.items():
            # Seleccionar canales
            ch_indices = [eeg_channels.index(ch) for ch in channels if ch in eeg_channels]
            
            if len(ch_indices) != len(channels):
                continue
            
            epoch_subset = epoch_data[ch_indices, :]
            
            # Calcular coherencia
            coh_matrix = compute_coherence_matrix(epoch_subset, sfreq, ALPHA_BAND)
            
            # Aplicar máscaras
            coh_matrix = apply_masks(coh_matrix)
            
            # Guardar
            filename = f"{subject}_epoch{epoch_idx:04d}.npy"
            save_path = os.path.join(OUTPUT_PATH, split, n_ch, label, filename)
            np.save(save_path, coh_matrix)
            
            matrices_count += 1
    
    return n_epochs

In [8]:
# Procesar Training - PD
print("=" * 60)
print("PROCESANDO TRAINING - PD")
print("=" * 60)

total_pd_train = 0
for subject in tqdm(PD_TRAIN, desc="PD Train"):
    n_epochs = process_subject(subject, 'PD', 'Training', CHANNEL_CONFIGS)
    total_pd_train += n_epochs

print(f"\nTotal epochs PD Training: {total_pd_train}")

PROCESANDO TRAINING - PD


PD Train:   0%|          | 0/20 [00:00<?, ?it/s]

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


PD Train:   5%|▌         | 1/20 [24:01<7:36:21, 1441.14s/it]

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


...

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


PD Train:  95%|█████████▌| 19/20 [5:03:50<15:41, 941.43s/it]

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


PD Train: 100%|██████████| 20/20 [5:18:38<00:00, 955.93s/it]


Total epochs PD Training: 6189


In [9]:
# Procesar Training - CTL
print("=" * 60)
print("PROCESANDO TRAINING - CTL")
print("=" * 60)

total_ctl_train = 0
for subject in tqdm(CTL_TRAIN, desc="CTL Train"):
    n_epochs = process_subject(subject, 'CTL', 'Training', CHANNEL_CONFIGS)
    total_ctl_train += n_epochs

print(f"\nTotal epochs CTL Training: {total_ctl_train}")

PROCESANDO TRAINING - CTL


CTL Train:   0%|          | 0/20 [00:00<?, ?it/s]

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


CTL Train:   5%|▌         | 1/20 [45:27<14:23:37, 2727.22s/it]

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


...

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


CTL Train:  95%|█████████▌| 19/20 [8:11:27<22:29, 1349.31s/it]

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


CTL Train: 100%|██████████| 20/20 [8:32:24<00:00, 1537.23s/it]


Total epochs CTL Training: 6095


In [10]:
# Procesar Test - PD
print("=" * 60)
print("PROCESANDO TEST - PD")
print("=" * 60)

total_pd_test = 0
for subject in tqdm(PD_TEST, desc="PD Test"):
    n_epochs = process_subject(subject, 'PD', 'Test', CHANNEL_CONFIGS)
    total_pd_test += n_epochs

print(f"\nTotal epochs PD Test: {total_pd_test}")

PROCESANDO TEST - PD


PD Test:   0%|          | 0/5 [00:00<?, ?it/s]

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


PD Test:  20%|██        | 1/5 [13:25<53:41, 805.45s/it]

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


...

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


PD Test:  80%|████████  | 4/5 [53:55<13:36, 816.83s/it]

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


PD Test: 100%|██████████| 5/5 [1:09:27<00:00, 833.41s/it]


Total epochs PD Test: 1435


In [13]:
# Procesar Test - CTL
print("=" * 60)
print("PROCESANDO TEST - CTL")
print("=" * 60)

total_ctl_test = 0
for subject in tqdm(CTL_TEST, desc="CTL Test"):
    n_epochs = process_subject(subject, 'CTL', 'Test', CHANNEL_CONFIGS)
    total_ctl_test += n_epochs

print(f"\nTotal epochs CTL Test: {total_ctl_test}")

PROCESANDO TEST - CTL


CTL Test:   0%|          | 0/5 [00:00<?, ?it/s]

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


CTL Test:  20%|██        | 1/5 [22:42<1:30:48, 1362.03s/it]

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


...

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


CTL Test:  80%|████████  | 4/5 [1:27:39<21:54, 1314.29s/it]

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


CTL Test: 100%|██████████| 5/5 [1:51:22<00:00, 1336.57s/it]


Total epochs CTL Test: 1409
